In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

sns.set_theme(style="whitegrid")
SEMILLA = 2025
random.seed(SEMILLA)
np.random.seed(SEMILLA)
print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# P.01 y P.02 son los DOS LADOS de un mismo pasillo. Las claves antiguas se conservan.
PARAMETROS = {
    "numero_pasillos": 2, "racks_por_pasillo": 30,        # claves antiguas: lados y racks por lado
    "numero_lados_pasillo": 2, "racks_por_lado": 30,
    "separacion_racks_m": 1.2, "ancho_pasillo_m": 2.5,
    "velocidad_picker_m_min": 60.0, "tiempo_pick_min": 15 / 60, "posiciones_por_carga": 20,
    "horas_ordinarias_semanales": 42, "trabajadores": 10,
    "replicas": 30
}
ESCENARIOS = {
    "Bajo":  {"cargas_noche": 50, "cargas_manana": 40, "cargas_tarde": 40},   # 130 cargas: escenario del documento
    "Medio": {"cargas_noche": 55, "cargas_manana": 45, "cargas_tarde": 45},
    "Alto":  {"cargas_noche": 60, "cargas_manana": 50, "cargas_tarde": 50}
}
HORAS_ORDINARIAS_TOTALES = PARAMETROS["horas_ordinarias_semanales"] * PARAMETROS["trabajadores"]
print("Parámetros cargados correctamente")
print(f"Lados del pasillo: {PARAMETROS['numero_lados_pasillo']} | racks por lado: {PARAMETROS['racks_por_lado']}")
print(f"Horas ordinarias semanales totales: {HORAS_ORDINARIAS_TOTALES}")
for n, d in ESCENARIOS.items():
    print(f"- {n}: {sum(d.values())} cargas semanales simuladas")

In [ ]:
# "pasillo" = LADO del pasillo (1 = P.01, 2 = P.02). Las coordenadas x e y se usan solo para dibujar.
def crear_ubicaciones(parametros):
    ubicaciones = []
    for pasillo in range(1, parametros["numero_pasillos"] + 1):
        for rack in range(1, parametros["racks_por_pasillo"] + 1):
            ubicaciones.append({
                "ubicacion": f"P{pasillo}-R{rack:02d}",
                "pasillo": pasillo,
                "lado": f"P.{pasillo:02d}",
                "rack": rack,
                "x_m": (rack - 1) * parametros["separacion_racks_m"],
                "y_m": (pasillo - 1) * parametros["ancho_pasillo_m"]
            })
    return pd.DataFrame(ubicaciones)

UBICACIONES = crear_ubicaciones(PARAMETROS)
print("Ubicaciones creadas correctamente")
print(f"Total de ubicaciones: {len(UBICACIONES)}")
display(UBICACIONES.head(10))

In [ ]:
ANCHO = PARAMETROS["ancho_pasillo_m"]
plt.figure(figsize=(14, 4.5))
for pasillo in sorted(UBICACIONES["pasillo"].unique()):
    datos = UBICACIONES[UBICACIONES["pasillo"] == pasillo]
    plt.plot(datos["x_m"], datos["y_m"], marker="o", linestyle="-", linewidth=2, markersize=5,
             label=f"Lado P.{pasillo:02d}")
    for _, f in datos.iterrows():
        if f["rack"] % 5 == 0 or f["rack"] == 1:
            plt.annotate(f"R{int(f['rack']):02d}", (f["x_m"], f["y_m"]), xytext=(0, 8),
                         textcoords="offset points", ha="center", fontsize=8)
plt.annotate("", xy=(17, 0), xytext=(17, ANCHO), arrowprops=dict(arrowstyle="<->"))
plt.text(17.4, ANCHO / 2, f"Ancho del pasillo: {ANCHO:.1f} m".replace(".", ","), va="center", fontsize=9)
plt.xlabel("Distancia longitudinal (m)"); plt.ylabel("Ancho del pasillo (m)")
plt.yticks(sorted(UBICACIONES["y_m"].unique()),
           [f"Lado P.{p:02d}" for p in sorted(UBICACIONES["pasillo"].unique())])
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("fig1_layout.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
def generar_carga(ubicaciones, parametros, escenario, turno, replica, numero_carga):
    """20 ubicaciones al azar, sin repetición, entre las 60 (2 lados x 30 racks).
    Semilla única por escenario, réplica, turno y carga: no hay cargas repetidas."""
    indice_escenario = list(ESCENARIOS).index(escenario)
    indice_turno = list(ESCENARIOS[escenario]).index(turno)
    semilla_carga = (SEMILLA + indice_escenario * 10_000_000 + replica * 100_000
                     + indice_turno * 1_000 + numero_carga)
    muestra = ubicaciones.sample(n=parametros["posiciones_por_carga"], replace=False,
                                 random_state=semilla_carga).copy()
    muestra["escenario"] = escenario; muestra["turno"] = turno
    muestra["replica"] = replica;     muestra["carga"] = numero_carga
    return muestra

def generar_cargas_escenario(ubicaciones, parametros, escenario, cargas_por_turno, replica):
    cargas = [generar_carga(ubicaciones, parametros, escenario, turno, replica, n)
              for turno, total in cargas_por_turno.items() for n in range(1, total + 1)]
    return pd.concat(cargas, ignore_index=True) if cargas else pd.DataFrame()

# Escenario Bajo, réplica 1 (las 130 cargas del documento)
cargas_prueba = generar_cargas_escenario(UBICACIONES, PARAMETROS, "Bajo", ESCENARIOS["Bajo"], replica=1)
print("Cargas simuladas correctamente")
display(cargas_prueba.head(10))

In [ ]:
total_cargas = cargas_prueba[["turno", "carga"]].drop_duplicates().shape[0]
esperadas = sum(ESCENARIOS["Bajo"].values())
posiciones = PARAMETROS["posiciones_por_carga"]
print("Registros de picking:", len(cargas_prueba), "| cargas:", total_cargas)
display(cargas_prueba.groupby("turno")[["carga"]].nunique().rename(columns={"carga": "numero_cargas"}))
firmas = cargas_prueba.groupby(["turno", "carga"]).apply(lambda g: tuple(sorted(g["ubicacion"])))
print("Cargas distintas:", firmas.nunique(), "de", len(firmas))
assert total_cargas == esperadas, "El número de cargas no coincide con el escenario"
assert len(cargas_prueba) == esperadas * posiciones, "El número de registros no coincide"
assert firmas.nunique() == len(firmas), "Hay cargas repetidas"
print("OK: 130 cargas, todas distintas")

In [ ]:
SEP = PARAMETROS["separacion_racks_m"]
ANCHO = PARAMETROS["ancho_pasillo_m"]

def _tramos_metodo_actual(carga):
    """Lado P.01 completo (ida recogiendo y retorno con carga), cruce en el muelle, lado P.02 completo."""
    lados = [carga.loc[carga["pasillo"] == 1, "rack"].tolist(),
             carga.loc[carga["pasillo"] == 2, "rack"].tolist()]
    tramos, n = [], 0
    for k, racks in enumerate(lados):
        if not racks:
            continue
        if k == 1 and lados[0]:
            tramos.append((ANCHO, n))
        pos = 0.0
        for r in sorted(racks):
            tramos.append((r * SEP - pos, n)); pos = r * SEP; n += 1
        tramos.append((pos, n))
    return tramos

def _tramos_ri_rsp(carga):
    """Desde el rack más lejano hacia el muelle, intercalando los dos lados (cruce diagonal)."""
    pk = sorted(zip(carga["rack"], carga["pasillo"]), key=lambda x: (-x[0], x[1]))
    tramos, n = [(pk[0][0] * SEP, 0)], 1
    for (r0, s0), (r1, s1) in zip(pk, pk[1:]):
        dx = (r0 - r1) * SEP; dy = ANCHO if s0 != s1 else 0.0
        tramos.append(((dx ** 2 + dy ** 2) ** 0.5, n)); n += 1
    tramos.append((pk[-1][0] * SEP, n))
    return tramos

def ordenar_ruta_metodo_actual(carga):
    ruta = carga.sort_values(by=["pasillo", "rack"], ascending=[True, True]).reset_index(drop=True)
    ruta.attrs["metodo"] = "actual"; return ruta

def ordenar_ruta_ri_rsp(carga):
    ruta = carga.sort_values(by=["rack", "pasillo"], ascending=[False, True]).reset_index(drop=True)
    ruta.attrs["metodo"] = "ri_rsp"; return ruta

def _tramos(ruta):
    m = ruta.attrs.get("metodo")
    if m == "actual": return _tramos_metodo_actual(ruta)
    if m == "ri_rsp": return _tramos_ri_rsp(ruta)
    raise ValueError("Genera la ruta con ordenar_ruta_metodo_actual u ordenar_ruta_ri_rsp")

def calcular_distancia_ruta(ruta):
    return 0.0 if ruta.empty else sum(d for d, _ in _tramos(ruta))

def calcular_metricas_ruta(ruta):
    """(distancia total, distancia con carga, pick·m). Con carga = todo tramo posterior al primer pick."""
    t = _tramos(ruta)
    return sum(d for d, _ in t), sum(d for d, c in t if c > 0), sum(d * c for d, c in t)

# Prueba de control: carga de la Tabla 10 del documento
P1 = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]; P2 = [2, 5, 8, 11, 14, 17, 20, 22, 26, 28]
prueba = pd.DataFrame([{"pasillo": 1, "rack": r} for r in P1] + [{"pasillo": 2, "rack": r} for r in P2])
a = calcular_metricas_ruta(ordenar_ruta_metodo_actual(prueba))
r = calcular_metricas_ruta(ordenar_ruta_ri_rsp(prueba))
print("Control  actual:", [round(x, 2) for x in a], "| RI-RSP:", [round(x, 2) for x in r])
assert abs(a[0] - 141.70) < .01 and abs(r[0] - 97.32) < .01
assert abs(a[1] - 138.10) < .01 and abs(r[1] - 61.32) < .01
print("OK: el código coincide con las Tablas 12 a 16 del documento")

In [ ]:
v_m_s = PARAMETROS["velocidad_picker_m_min"] / 60
t_pick_s = PARAMETROS["tiempo_pick_min"] * 60
filas = []
for (turno, numero_carga), carga in cargas_prueba.groupby(["turno", "carga"], dropna=False):
    d_a, c_a, m_a = calcular_metricas_ruta(ordenar_ruta_metodo_actual(carga))
    d_r, c_r, m_r = calcular_metricas_ruta(ordenar_ruta_ri_rsp(carga))
    picks = len(carga); ahorro = d_a - d_r
    filas.append({
        "turno": turno, "carga": numero_carga, "picks": picks,
        "distancia_actual_m": d_a, "distancia_ri_rsp_m": d_r, "ahorro_m": ahorro,
        "ahorro_porcentaje": (ahorro / d_a * 100) if d_a > 0 else 0.0,
        "distancia_con_carga_actual_m": c_a, "distancia_con_carga_ri_rsp_m": c_r,
        "pickm_actual": m_a, "pickm_ri_rsp": m_r,
        "ciclo_actual_s": d_a / v_m_s + picks * t_pick_s,
        "ciclo_ri_rsp_s": d_r / v_m_s + picks * t_pick_s})
resultados_rutas = (pd.DataFrame(filas).sort_values(by=["turno", "carga"]).reset_index(drop=True))
resultados_rutas.to_csv("resultados_rutas_final.csv", index=False, encoding="utf-8-sig")

assert len(resultados_rutas) == sum(ESCENARIOS["Bajo"].values())
assert (resultados_rutas["picks"] == PARAMETROS["posiciones_por_carga"]).all()
r = resultados_rutas
print(f"Cargas: {len(r)} | Reducción global de distancia: {100*r.ahorro_m.sum()/r.distancia_actual_m.sum():.2f} %")
print(f"Cargas donde RI-RSP NO ahorra distancia: {(r.ahorro_m <= 0).sum()} de {len(r)}")
display(r.head(10))

In [ ]:
distancia_total_actual = resultados_rutas["distancia_actual_m"].sum()
distancia_total_ri_rsp = resultados_rutas["distancia_ri_rsp_m"].sum()
ahorro_total_m = resultados_rutas["ahorro_m"].sum()
ahorro_total_porcentaje = (ahorro_total_m / distancia_total_actual * 100) if distancia_total_actual > 0 else 0.0
distancia_promedio_actual = resultados_rutas["distancia_actual_m"].mean()
distancia_promedio_ri_rsp = resultados_rutas["distancia_ri_rsp_m"].mean()
ahorro_promedio_m = resultados_rutas["ahorro_m"].mean()
ahorro_promedio_porcentaje = resultados_rutas["ahorro_porcentaje"].mean()

print("INDICADORES GENERALES DEL ESCENARIO")
print("=" * 50)
print(f"Total de cargas analizadas: {len(resultados_rutas)}")
print(f"Total de posiciones de picking: {len(cargas_prueba)}")
print("\nDistancia total:")
print(f"  Método actual: {distancia_total_actual:,.2f} m")
print(f"  RI-RSP:        {distancia_total_ri_rsp:,.2f} m")
print("\nAhorro:")
print(f"  Ahorro total:   {ahorro_total_m:,.2f} m")
print(f"  Ahorro global:  {ahorro_total_porcentaje:.2f}%")
print("\nPromedios por carga:")
print(f"  Método actual:  {distancia_promedio_actual:,.2f} m")
print(f"  RI-RSP:         {distancia_promedio_ri_rsp:,.2f} m")
print(f"  Ahorro:         {ahorro_promedio_m:,.2f} m")
print(f"  Ahorro %:       {ahorro_promedio_porcentaje:.2f}%")

In [ ]:
columnas_requeridas = ["distancia_actual_m", "distancia_ri_rsp_m", "ahorro_m", "ahorro_porcentaje"]
faltantes = [c for c in columnas_requeridas if c not in resultados_rutas.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en resultados_rutas: {faltantes}")
if resultados_rutas.empty:
    raise ValueError("La tabla resultados_rutas está vacía.")

distancias_actuales = resultados_rutas["distancia_actual_m"].astype(float)
distancias_ri_rsp = resultados_rutas["distancia_ri_rsp_m"].astype(float)
ahorros = resultados_rutas["ahorro_m"].astype(float)
porcentajes_ahorro = resultados_rutas["ahorro_porcentaje"].astype(float)

diferencia_maxima = np.abs(ahorros - (distancias_actuales - distancias_ri_rsp)).max()
if diferencia_maxima > 0.01:
    print("Advertencia: el ahorro registrado no coincide con el recalculado.")

cargas_mejoran = (ahorros > 1e-9).sum()
cargas_iguales = np.isclose(ahorros, 0, atol=1e-9).sum()
cargas_empeoran = (ahorros < -1e-9).sum()
total_cargas = len(resultados_rutas)
porcentaje_cargas_mejoran = cargas_mejoran / total_cargas * 100
porcentaje_cargas_iguales = cargas_iguales / total_cargas * 100
porcentaje_cargas_empeoran = cargas_empeoran / total_cargas * 100

resumen_estadistico = pd.DataFrame({
    "Indicador": ["Distancia actual promedio (m)", "Distancia RI-RSP promedio (m)", "Ahorro promedio (m)",
                  "Ahorro promedio (%)", "Ahorro mínimo (m)", "Ahorro máximo (m)",
                  "Desviación estándar del ahorro (m)", "Mediana del ahorro (m)",
                  "Cargas con mejora", "Cargas sin cambio", "Cargas con aumento de distancia",
                  "Porcentaje de cargas con mejora (%)", "Porcentaje de cargas sin cambio (%)",
                  "Porcentaje de cargas con aumento (%)"],
    "Valor": [distancias_actuales.mean(), distancias_ri_rsp.mean(), ahorros.mean(), porcentajes_ahorro.mean(),
              ahorros.min(), ahorros.max(), ahorros.std(), ahorros.median(),
              cargas_mejoran, cargas_iguales, cargas_empeoran,
              porcentaje_cargas_mejoran, porcentaje_cargas_iguales, porcentaje_cargas_empeoran]})
resumen_estadistico["Valor"] = resumen_estadistico["Valor"].round(2)

print("ANÁLISIS ESTADÍSTICO Y VALIDACIÓN")
print("=" * 60)
print(f"Total de cargas evaluadas: {total_cargas}")
print(f"Diferencia máxima entre ahorro registrado y recalculado: {diferencia_maxima:.4f} m")
print(f"  Cargas con mejora:         {cargas_mejoran} ({porcentaje_cargas_mejoran:.2f}%)")
print(f"  Cargas sin cambio:         {cargas_iguales} ({porcentaje_cargas_iguales:.2f}%)")
print(f"  Cargas con mayor distancia: {cargas_empeoran} ({porcentaje_cargas_empeoran:.2f}%)")
display(resumen_estadistico)

In [ ]:
a = resultados_rutas["distancia_actual_m"].astype(float)
b = resultados_rutas["distancia_ri_rsp_m"].astype(float)
dif = a - b
sw, t, w = stats.shapiro(dif), stats.ttest_rel(a, b), stats.wilcoxon(a, b)
ic = stats.t.interval(0.95, len(dif) - 1, loc=dif.mean(), scale=stats.sem(dif))
d_cohen = dif.mean() / dif.std(ddof=1)

tabla_pruebas = pd.DataFrame({
    "Prueba": ["Shapiro-Wilk (normalidad de las diferencias)", "t pareada", "Wilcoxon",
               "IC 95 % del ahorro medio (m)", "d de Cohen (pareado)"],
    "Resultado": [f"W = {sw.statistic:.3f}; p = {sw.pvalue:.3g}",
                  f"t = {t.statistic:.2f}; gl = {len(dif)-1}; p = {t.pvalue:.3g}",
                  f"W = {w.statistic:.1f}; p = {w.pvalue:.3g}",
                  f"[{ic[0]:.2f}; {ic[1]:.2f}]", f"{d_cohen:.2f}"]})
display(tabla_pruebas)

In [ ]:
NOMBRES_TURNO = {"cargas_manana": "Mañana", "cargas_tarde": "Tarde", "cargas_noche": "Noche"}
ORDEN_TURNOS = ["Mañana", "Tarde", "Noche"]
resultados_rutas["turno_es"] = resultados_rutas["turno"].map(NOMBRES_TURNO)

resultados_por_turno = (resultados_rutas.groupby(["turno", "turno_es"], as_index=False).agg(
    cargas=("carga", "count"), posiciones_picking=("picks", "sum"),
    distancia_actual_total_m=("distancia_actual_m", "sum"),
    distancia_ri_rsp_total_m=("distancia_ri_rsp_m", "sum"),
    ahorro_total_m=("ahorro_m", "sum"), ahorro_promedio_m=("ahorro_m", "mean"),
    ahorro_promedio_porcentaje=("ahorro_porcentaje", "mean")))
resultados_por_turno["ahorro_global_porcentaje"] = (
    resultados_por_turno["ahorro_total_m"] / resultados_por_turno["distancia_actual_total_m"] * 100)
resultados_por_turno["porcentaje_de_cargas"] = (
    resultados_por_turno["cargas"] / resultados_por_turno["cargas"].sum() * 100)
resultados_por_turno["turno_es"] = pd.Categorical(resultados_por_turno["turno_es"],
                                                  categories=ORDEN_TURNOS, ordered=True)
resultados_por_turno = resultados_por_turno.sort_values("turno_es").reset_index(drop=True)
cols = resultados_por_turno.select_dtypes(include="number").columns
resultados_por_turno[cols] = resultados_por_turno[cols].round(2)
display(resultados_por_turno)

grupos = [g["ahorro_porcentaje"].values for _, g in resultados_rutas.groupby("turno_es", observed=True)]
kw = stats.kruskal(*grupos)
print(f"Kruskal-Wallis del ahorro % entre turnos: H = {kw.statistic:.2f}, p = {kw.pvalue:.3g}")

In [ ]:
from matplotlib.ticker import FuncFormatter
datos_graficas = resultados_por_turno.copy()
datos_graficas["turno_es"] = datos_graficas["turno_es"].astype(str)
etiquetas = datos_graficas["turno_es"].tolist()
colores = {"actual": "#4472C4", "ri_rsp": "#70AD47", "ahorro": "#ED7D31"}
miles = lambda x, pos=None: f"{x:,.0f}".replace(",", ".")
dec2 = lambda x: f"{x:.2f}".replace(".", ",")

def etiquetar(ax, barras, formato):
    for b in barras:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, h, formato(h), ha="center", va="bottom", fontsize=9)

fig, ax = plt.subplots(figsize=(7, 4.2))
pos = np.arange(len(datos_graficas)); ancho = 0.35
b1 = ax.bar(pos - ancho / 2, datos_graficas["distancia_actual_total_m"], ancho, label="Método actual", color=colores["actual"])
b2 = ax.bar(pos + ancho / 2, datos_graficas["distancia_ri_rsp_total_m"], ancho, label="RI-RSP", color=colores["ri_rsp"])
ax.set_xlabel("Turno"); ax.set_ylabel("Distancia total recorrida (m)")
ax.set_xticks(pos); ax.set_xticklabels(etiquetas); ax.yaxis.set_major_formatter(FuncFormatter(miles))
ax.legend(); ax.grid(axis="y", alpha=0.3); etiquetar(ax, b1, miles); etiquetar(ax, b2, miles)
plt.tight_layout(); plt.savefig("fig2_distancia_por_turno.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.2))
b = ax.bar(etiquetas, datos_graficas["ahorro_total_m"], color=colores["ahorro"])
ax.set_xlabel("Turno"); ax.set_ylabel("Ahorro acumulado de distancia (m)")
ax.yaxis.set_major_formatter(FuncFormatter(miles)); ax.grid(axis="y", alpha=0.3); etiquetar(ax, b, miles)
plt.tight_layout(); plt.savefig("fig3_ahorro_acumulado_por_turno.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.2))
b = ax.bar(etiquetas, datos_graficas["ahorro_global_porcentaje"], color=colores["ri_rsp"])
ax.set_xlabel("Turno"); ax.set_ylabel("Ahorro global de distancia (%)"); ax.grid(axis="y", alpha=0.3)
etiquetar(ax, b, lambda h: dec2(h) + " %")
plt.tight_layout(); plt.savefig("fig4_porcentaje_ahorro_por_turno.png", dpi=300); plt.show()

In [ ]:
SMLMV = 1_750_905                                    # SMLMV 2026
VALOR_HORA = round(SMLMV / 210)                      # 210 h mensuales = 42 h / 6 días x 30 días  -> 8.338
VALOR_HORA_EXTRA = int(VALOR_HORA * 1.25 + 0.5)      # recargo diurno del 25 %                    -> 10.423
HORAS_EXTRA_POR_AUXILIAR = 8                         # supuesto del documento (h por semana)
SEMANAS_MES, SEMANAS_ANIO = 4.33, 52

he_actual = HORAS_EXTRA_POR_AUXILIAR * PARAMETROS["trabajadores"]            # 80 h/semana
carga_total = HORAS_ORDINARIAS_TOTALES + he_actual                           # 500 h/semana
red_ciclo = 1 - resultados_rutas["ciclo_ri_rsp_s"].sum() / resultados_rutas["ciclo_actual_s"].sum()
tiempo_liberado = carga_total * red_ciclo
he_ri_rsp = max(0.0, he_actual - tiempo_liberado)

def costos(h):
    sem = h * VALOR_HORA_EXTRA
    return round(sem), round(sem * SEMANAS_MES), round(sem * SEMANAS_ANIO)

ca, cr = costos(he_actual), costos(he_ri_rsp)
tabla_horas_extra = pd.DataFrame({
    "Indicador": ["Reducción del tiempo de ciclo (%)", "Horas extra por semana (h)",
                  "Costo por semana ($)", "Costo por mes, 4,33 semanas ($)", "Costo por año, 52 semanas ($)"],
    "Método actual": [None, round(he_actual, 1), ca[0], ca[1], ca[2]],
    "RI-RSP": [round(100 * red_ciclo, 2), round(he_ri_rsp, 1), cr[0], cr[1], cr[2]],
    "Ahorro": [None, round(he_actual - he_ri_rsp, 1), ca[0] - cr[0], ca[1] - cr[1], ca[2] - cr[2]]})
print(f"Valor hora ordinaria: ${VALOR_HORA:,} | hora extra diurna: ${VALOR_HORA_EXTRA:,}".replace(",", "."))
print(f"Reducción de horas extra: {100 * (1 - he_ri_rsp / he_actual):.1f} %")
display(tabla_horas_extra)

In [ ]:
v_m_s = PARAMETROS["velocidad_picker_m_min"] / 60
t_pick_s = PARAMETROS["tiempo_pick_min"] * 60
filas_rep = []
for rep in range(1, PARAMETROS["replicas"] + 1):
    cargas_rep = generar_cargas_escenario(UBICACIONES, PARAMETROS, "Bajo", ESCENARIOS["Bajo"], rep)
    da = dr = 0.0; n_cargas = 0
    for _, carga in cargas_rep.groupby(["turno", "carga"]):
        da += calcular_distancia_ruta(ordenar_ruta_metodo_actual(carga))
        dr += calcular_distancia_ruta(ordenar_ruta_ri_rsp(carga)); n_cargas += 1
    ciclo_a = da / v_m_s + n_cargas * PARAMETROS["posiciones_por_carga"] * t_pick_s
    ciclo_r = dr / v_m_s + n_cargas * PARAMETROS["posiciones_por_carga"] * t_pick_s
    filas_rep.append({"replica": rep, "red_distancia_pct": 100 * (1 - dr / da), "red_ciclo_pct": 100 * (1 - ciclo_r / ciclo_a)})
tabla_replicas = pd.DataFrame(filas_rep)

def resumen(col):
    x = tabla_replicas[col]; ic = stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    return [round(x.mean(), 2), round(x.std(ddof=1), 2), round(x.min(), 2), round(x.max(), 2), f"[{ic[0]:.2f}; {ic[1]:.2f}]"]

resumen_replicas = pd.DataFrame({"Estadístico": ["Media", "Desv. estándar", "Mínimo", "Máximo", "IC 95 %"],
                                 "Reducción de distancia (%)": resumen("red_distancia_pct"),
                                 "Reducción del tiempo de ciclo (%)": resumen("red_ciclo_pct")})
display(resumen_replicas)

In [ ]:
with pd.ExcelWriter("informe_rutas_RI_RSP.xlsx") as escritor:
    resumen_general = pd.DataFrame({
        "Indicador": ["Total de cargas", "Posiciones de picking", "Distancia total actual (m)",
                      "Distancia total RI-RSP (m)", "Ahorro total (m)", "Ahorro global (%)"],
        "Valor": [len(resultados_rutas), len(cargas_prueba), distancia_total_actual,
                  distancia_total_ri_rsp, ahorro_total_m, ahorro_total_porcentaje]})
    resumen_general["Valor"] = resumen_general["Valor"].round(2)
    resumen_general.to_excel(escritor, sheet_name="Resumen general", index=False)
    resultados_por_turno.to_excel(escritor, sheet_name="Resultados por turno", index=False)
    resultados_rutas.to_excel(escritor, sheet_name="Resultados por carga", index=False)
    resumen_estadistico.to_excel(escritor, sheet_name="Analisis estadistico", index=False)
    tabla_pruebas.to_excel(escritor, sheet_name="Prueba pareada", index=False)
    tabla_horas_extra.to_excel(escritor, sheet_name="Horas extra", index=False)
    if "resumen_replicas" in globals():
        resumen_replicas.to_excel(escritor, sheet_name="30 replicas", index=False)
print("Informe final generado: informe_rutas_RI_RSP.xlsx")

In [ ]:
print("=== RESUMEN PARA ENVIAR ===")
print(f"Semilla: {SEMILLA} | escenario Bajo: {ESCENARIOS['Bajo']}")
print(f"Cargas: {len(resultados_rutas)} | posiciones: {len(cargas_prueba)}")
print(f"Distancia total actual: {distancia_total_actual:.1f} m | RI-RSP: {distancia_total_ri_rsp:.1f} m")
print(f"Ahorro total: {ahorro_total_m:.1f} m | global: {ahorro_total_porcentaje:.2f} % | promedio %: {ahorro_promedio_porcentaje:.2f}")
print(f"Promedio por carga: actual {distancia_promedio_actual:.1f} m | RI-RSP {distancia_promedio_ri_rsp:.1f} m")
r = resultados_rutas
print(f"Ciclo promedio: actual {r.ciclo_actual_s.mean():.1f} s | RI-RSP {r.ciclo_ri_rsp_s.mean():.1f} s")
print(f"Distancia con carga: actual {r.distancia_con_carga_actual_m.sum():.1f} m | RI-RSP {r.distancia_con_carga_ri_rsp_m.sum():.1f} m")
print(f"Pick·m: actual {r.pickm_actual.sum():.1f} | RI-RSP {r.pickm_ri_rsp.sum():.1f} | reducción {100*(1-r.pickm_ri_rsp.sum()/r.pickm_actual.sum()):.1f} %")
print(f"Cargas: mejoran {cargas_mejoran} | iguales {cargas_iguales} | empeoran {cargas_empeoran}")
print("\n-- Estadísticos --");   print(resumen_estadistico.to_string(index=False))
print("\n-- Prueba pareada --"); print(tabla_pruebas.to_string(index=False))
print("\n-- Por turno --");      print(resultados_por_turno.to_string(index=False))
print(f"Kruskal-Wallis entre turnos: H = {kw.statistic:.2f}, p = {kw.pvalue:.3g}")
print("\n-- Horas extra --");    print(tabla_horas_extra.to_string(index=False))
if "resumen_replicas" in globals():
    print("\n-- 30 réplicas --"); print(resumen_replicas.to_string(index=False))